# Main file 

05.11.2025

Written by: Arnav Kumar Upadhyay, Cedric Meyer adapted from MATLAB code of Marc, Jan Spindler DOI-https://doi.org/10.1021/acssynbio.4c00578

This script is the main script for the TXTL v2 model in INDEPENDENT GENE SIMULATION mode.

The script simulates each gene independently without competition for shared resources.


## Settings

In [ ]:
# Packages functions and data types --------
include("packages.jl") 


# Sequences 
sequence_file	= "lacS.fa"
hybrid_file		= "lacS.txt"

# initialization of data containers
scale = txtl_types.init_scale()         # struct that makes it possible to influence rates by scaling them
µ = 1                    # A growth rate (0.1 to 1.8) can be chosen and the proteins of the TXTL machinery will be tuned according to that growth rate and the processivity of the RNA polymerase


## flags for efficiency factors
positive_aminoacid	= true 
anti_shinedalgarno	= false      # If no hybridization file is available put flag to false. We apply the intaRNA program’s intarna function (q: aSD, t: sequence, SeedBP: 5, out: tMinE) to generate a hybridization profile along the sequence. Link to the website: https://rna.informatik.uni-freiburg.de/IntaRNA/Input.jsp;jsessionid=2AE257E91E7276283F22CD77A5A8AA03
dynamic_efficiency	= true 

flags = txtl_types.flags(positive_aminoacid, anti_shinedalgarno, dynamic_efficiency)


## Import information about model species and sequences 
seqs, seq_names, hyb_file_dict, species_and_seqs, idx = species_and_sequences(µ, sequence_file, hybrid_file, flags)


## Calculate efficiency factors 
efac_pos_aa_dict, efac_int_sd_dict, dynamic_efficiencies = efficiency_factor_initialization(seqs, seq_names, hyb_file_dict, species_and_seqs)



## calculation of initial concentrations
y0, initial_concentrations = calculation_of_initial_concentration(species_and_seqs, seqs, seq_names, scale, idx);



## calculate and load kinetic parameters
p_met, p_seq, p_dyn = parameters(initial_concentrations, species_and_seqs);



## Release factor balance
RF_balance!(y0, initial_concentrations)



# Preintegration
## Definition of mass matrix
conserved = [idx.EFG, idx.EFD] # EFG, EFTu_GDP - Conserved species, Algebraic equations are given by mass balances
M = Matrix{Float64}(I, length(y0), length(y0)) # Creates unity matrix in the size of y0
M[conserved, :] .= 0; # setting mass matrix to zero to  remove time derivatives from those variables. This converts these differential equations into algebraic constraints

tags_pre = txtl_types.data_struct_preintegration(scale, idx, species_and_seqs, flags, dynamic_efficiencies, initial_concentrations, p_met, p_seq, p_dyn)

## Integration of the DAE system without DNA in the system 
y_steady_state = preintegration(tags_pre, y0, M);



# Elongation factor balance 
EF_balance!(tags_pre, y0, y_steady_state)



# Build states vector for multiple sequences 
y0_all = combined_states_vector(tags_pre, y_steady_state, seqs, seq_names)

# Prepare per-sequence tags (for use in ODE)
local_tags_list = generate_local_tags(seqs, seq_names, tags_pre, efac_pos_aa_dict, efac_int_sd_dict);


# Integration
M = Matrix{Float64}(I, length(y0_all), length(y0_all))
M[conserved, :] .= 0

tspan 	= (0.0, 100.0) ## Define time span for the simulation
p_parallel = local_tags_list
f 		= ODEFunction(ODE_parallel_distributed!, mass_matrix = M) ## Define ODEFunction to make use of mass matrix
prob 	= ODEProblem(f, y0_all, tspan, p_parallel); ## Create an ODE problem



# --- Timing the integration step ---
println("\n⏳ Integrating ODE system (distributed, test set)...")

sol = solve(prob, FBDF(autodiff=false), reltol = 1e-10, abstol = 1e-6); ## Solve ODE system


✅ Successfully loaded 8 sequences

⏳ Integrating ODE system (distributed, test set)...
241
103
2
here
3
here
4
here
5
here
6
here
7
here
8
here
9
here
10
here
11
here
12
here
13
here
14
here
15
here
16
here
17
here
18
here
19
here
20
here
21
here
22
here
23
here
24
here
25
here
26
here
27
here
28
here
29
here
30
here
31
here
32
here
33
here
34
here
35
here
36
here
37
here
38
here
39
here
40
here
41
here
42
here
43
here
44
here
45
here
46
here
47
here
48
here
49
here
50
here
51
here
52
here
53
here
54
here
55
here
56
here
57
here
58
here
59
here
60
here
61
here
62
here
63
here
64
here
65
here
66
here
67
here
68
here
69
here
70
here
71
here
72
here
73
here
74
here
75
here
76
here
77
here
78
here
79
here
80
here
81
here
82
here
83
here
84
here
85
here
86
here
87
here
88
here
89
here
90
here
91
here
rplU
241
103
2
here
3
here
4
here
5
here
6
here
7
here
8
here
9
here
10
here
11
here
12
here
13
here
14
here
15
here
16
here
17
here
18
here
19
here
20
here
21
here
22
here
23
here
24
here
25
h

LoadError: BoundsError: attempt to access 103-element view(::Vector{Float64}, 139:241) with eltype Float64 at index [92:104]

In [45]:
for i in eachindex(local_tags_list)
    println(local_tags_list[i].sequence_name)
end

rplU
fusA
ompF
tufA
rpmJ
rplE
tufB
Eco_lacS


In [48]:
local_tags_list[1].idx_first_codon

139

In [47]:
local_tags_list[1].Tags

138-element Vector{SubString{String}}:
 "S30"
 "S50"
 "IF1"
 "IF2"
 "IF3"
 "SC1"
 "SC2"
 "SC3"
 "SC4"
 "PIC"
 "EFG"
 "EFD"
 "EGT"
 ⋮
 "t1t"
 "t2t"
 "t4t"
 "v1t"
 "v2t"
 "w1t"
 "y1t"
 "RNA1"
 "S701"
 "rna1"
 "PRT1"
 "PTC1"

In [46]:
local_tags_list[1].species_and_sequences.Seq

104-element Vector{String}:
 "AUG"
 "UAC"
 "GCG"
 "GUU"
 "UUC"
 "CAA"
 "AGU"
 "GGU"
 "GGU"
 "AAA"
 "CAA"
 "CAC"
 "CGA"
 ⋮
 "UUC"
 "ACU"
 "GAU"
 "GUG"
 "AAA"
 "AUU"
 "ACU"
 "GGC"
 "AUC"
 "AGC"
 "GCC"
 "UAA"